In [1]:
import pandas as pd
import sqlite3
import os
from tqdm import tqdm

In [7]:
def load_battery_data_to_db(base_path, db_name="battery_factory.db"):
    # 1. DB 연결 (없으면 생성)
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    
    print(f"🚀 DB 적재 시작: {db_name}")
    
    # 2. 적재할 폴더 리스트 (train, test)
    data_dirs = [
        os.path.join(base_path, 'train'),
        os.path.join(base_path, 'test')
    ]
    
    first_file = True
    
    for data_dir in data_dirs:
        if not os.path.exists(data_dir):
            continue
            
        print(f"\n📂 현재 폴더 처리 중: {data_dir}")
        files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
        
        for file_name in tqdm(files, desc="파일 적재 중"):
            file_path = os.path.join(data_dir, file_name)
            
            # 파일명에서 SerialNumber와 타입(chg/dchg) 추출 [cite: 197, 201]
            # 예: 1000_chg.csv -> Serial: 1000, Type: chg
            sn_part = file_name.split('.')[0]
            serial_number = sn_part.split('_')[0]
            test_type = sn_part.split('_')[1] if '_' in sn_part else 'unknown'
            
            try:
                # 가이드북 인코딩(cp949) 반영하여 데이터 로드 [cite: 175]
                # 메모리 효율을 위해 chunksize 사용 (대용량 대응) 
                chunks = pd.read_csv(file_path, encoding='cp949', chunksize=50000)
                
                for chunk in chunks:
                    chunk['SerialNumber'] = serial_number
                    chunk['TestType'] = test_type
                    
                    # 3. DB에 적재 (첫 파일이면 테이블 생성, 이후엔 append)
                    if first_file:
                        chunk.to_sql('battery_logs', conn, if_exists='replace', index=False)
                        first_file = False
                    else:
                        chunk.to_sql('battery_logs', conn, if_exists='append', index=False)
                        
            except Exception as e:
                print(f"❌ 파일 처리 실패 ({file_name}): {e}")

    # 4. 성능 최적화: 인덱스 생성 (SerialNumber와 Time으로 조회할 일이 많음) [cite: 433, 586]
    print("\n⚡ 검색 최적화를 위한 인덱스 생성 중...")
    cursor.execute("CREATE INDEX idx_serial_time ON battery_logs (SerialNumber, Time);")
    conn.commit()
    conn.close()
    print("✅ 모든 데이터 적재 및 인덱싱 완료!")

# 실행 (경로는 태형님의 트리 구조에 맞게 수정)
if __name__ == "__main__":
    base_data_path = "/home/jovyan/project/data/data/raw_data" 
    load_battery_data_to_db(base_data_path)

🚀 DB 적재 시작: battery_factory.db
/home/jovyan/project/data/data/raw_data/train

📂 현재 폴더 처리 중: /home/jovyan/project/data/data/raw_data/train


파일 적재 중: 100%|██████████| 102/102 [00:26<00:00,  3.88it/s]


/home/jovyan/project/data/data/raw_data/test

📂 현재 폴더 처리 중: /home/jovyan/project/data/data/raw_data/test


파일 적재 중: 100%|██████████| 9/9 [00:01<00:00,  6.96it/s]



⚡ 검색 최적화를 위한 인덱스 생성 중...
✅ 모든 데이터 적재 및 인덱싱 완료!
